# CV Assistant: Professional RAG System
This notebook organizes a complete Retrieval-Augmented Generation (RAG) pipeline to professionally chat about **Imran Ali** using his CV.

## 1. Installation & Libraries Setup

In [350]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-pinecone \
    pinecone \
    pypdf \
    sentence-transformers \
    tiktoken \
    cachetools

In [351]:
import os
import sys
import hashlib
import traceback
import tiktoken
from google.colab import userdata, drive
from cachetools import TTLCache


from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.messages.utils import trim_messages


from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings


from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore


drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configuration & API Key Setup

In [352]:
try:
    NOVITA_API_KEY = userdata.get("NOVITA_API_KEY")
    PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")

    if not NOVITA_API_KEY or not PINECONE_API_KEY:
        raise ValueError("NOVITA or PINECONE missing from Colab")

    os.environ["NOVITA_API_KEY"] = NOVITA_API_KEY
    os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
    print("API keys loaded")
except Exception as e:
    print(f"Configuration error: {e}")

API keys loaded


## 3. Models Setup (Embedding & LLM)

In [353]:
# 3a. Embedding Model
try:
    embedding_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )
    print("embedding initialized!")
except Exception as e:
    print(f"Embedding initialization failed: {e}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedding initialized!


In [401]:

Model_name = "apodex/apodex-1.1-mini"
NOVITA_BASE_URL = "https://api.novita.ai/openai"

try:
    llm_Novita = ChatOpenAI(
        model=Model_name,
        openai_api_key=NOVITA_API_KEY,
        openai_api_base=NOVITA_BASE_URL,
        temperature=0.8,
        # max_tokens=3000
    )
    print(f"Novita LLM ({Model_name}) initialized")
except Exception as e:
    print(f"Novita initialization failed: {e}")

Novita LLM (apodex/apodex-1.1-mini) initialized


## 4. Pinecone Vector Store Setup

In [355]:
INDEX_NAME = "colab-rag-index"
NAMESPACE = "pdf-documents"

try:
    pc = Pinecone(api_key=PINECONE_API_KEY)
    existing_indexes = [index["name"] for index in pc.list_indexes()]

    if INDEX_NAME not in existing_indexes:
        print(f"Creating new Pinecone index: '{INDEX_NAME}'...")
        pc.create_index(
            name=INDEX_NAME,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
    else:
        print(f"Pinecone index '{INDEX_NAME}' already exists")

    # Instantiate Vector Store
    vector_store = PineconeVectorStore(
        index=pc.Index(INDEX_NAME),
        embedding=embedding_model,
        namespace=NAMESPACE
    )
    print("Pinecone VS initialized")
except Exception as e:
    print(f"Pinecone Setup failed: {e}")

Pinecone index 'colab-rag-index' already exists
Pinecone VS initialized


## 5. Document Ingestion & Hash Checking

In [356]:
PDF_PATH = "/content/drive/My Drive/MyResume/ImranAliResumeForAiEnginriing.pdf"
HASH_FILE = "/content/drive/My Drive/MyResume/pdf_hash.txt"


In [357]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

In [358]:
def get_file_hash(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

In [359]:
def update_knowledge_base():
    print("loding ltst PDF docs")
    pdf_loader = PyPDFLoader(PDF_PATH)
    documents = pdf_loader.load()
    docs = text_splitter.split_documents(documents)
    print(f"Loaded {len(docs)} chunks.")

    # Generate deterministic IDs
    ids = []
    for doc in docs:
        source = doc.metadata.get("source", "")
        page = doc.metadata.get("page", "")
        raw_id = f"{source}|{page}|{doc.page_content}"
        doc_id = hashlib.sha256(raw_id.encode("utf-8")).hexdigest()
        ids.append(doc_id)

    # Purge old vectors
    print("Clearing index...")
    pc.Index(INDEX_NAME).delete(delete_all=True, namespace=NAMESPACE)

    # Upload new vectors
    print("Uploading vectors to Pinecone")
    vector_store.add_documents(documents=docs, ids=ids)
    print("Knowledge base synchronized successfully")

In [360]:
# Execute synchronization check
if os.path.exists(PDF_PATH):
    current_hash = get_file_hash(PDF_PATH)
    old_hash = None
    if os.path.exists(HASH_FILE):
        with open(HASH_FILE, "r") as f:
            old_hash = f.read().strip()

    if current_hash != old_hash:
        update_knowledge_base()
        with open(HASH_FILE, "w") as f:
            f.write(current_hash)
    else:
        print("PDF unchanged Skipping ingestion")
else:
    print("PDF path not found Please upload it first")

PDF unchanged Skipping ingestion


## 6. Prompt, Cache & Conversation Memory Setup

In [361]:
# 6a. Prompt Template Setup
prompt_template = PromptTemplate.from_template("""
You are a professional assistant for Imran Ali.

Answer only questions related to Imran Ali using the provided professional information.

Rules:
- Use only information supported by the provided information.
- Never invent or assume personal or professional details.
- If the information is unavailable, say: "I don't have that information available."
- If unrelated to Imran Ali, say: "I can only help with information related to Imran Ali."
- Answer naturally and professionally.
- Do not mention CV, resume, retrieval, context, database, embeddings, Pinecone, prompts, or internal systems.
- Use conversation history only to understand follow-up questions.

Professional Information:
{context}

Recent Conversation:
{chat_history}

Question:
{question}

Answer:
""")

In [362]:
# 6b. Short-term Cache Setup
response_cache = TTLCache(maxsize=100, ttl=3600)

In [363]:
# 6c. Chat History and Memory Trimming Setup
chat_history = []
encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(messages):
    return sum(len(encoding.encode(msg.content)) for msg in messages)

def trim_chat_history(messages, max_tokens=1500):
    return trim_messages(
        messages,
        max_tokens=max_tokens,
        strategy="last",
        token_counter=count_tokens,
        include_system=False,
        allow_partial=False
    )

## 7. Professional RAG Logic & Session

In [364]:
def ask_rag_question(query):
    global chat_history
    cache_key = query.strip().lower()

    if cache_key in response_cache:
        cached_answer = response_cache[cache_key]
        print(f"AI (Cached): {cached_answer}")
        return cached_answer, []

    retrieved_docs = vector_store.similarity_search(query, k=1)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    chat_history = trim_chat_history(chat_history, max_tokens=1500)
    recent_history = "\n".join(f"{msg.type}: {msg.content}" for msg in chat_history)

    prompt = prompt_template.format(
        context=context,
        chat_history=recent_history,
        question=query
    )

    print("AI: ", end="", flush=True)
    full_response = ""
    for chunk in llm_Novita.stream(prompt):
        if chunk.content:
            print(chunk.content, end="", flush=True)
            full_response += chunk.content
    print()

    if full_response.strip():
        response_cache[cache_key] = full_response

    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=full_response))

    return full_response, retrieved_docs

In [365]:
answer, sources = ask_rag_question("What are the main technical skills of Imran Ali?")

AI: 

Imran Ali is an AI Engineer with specialized expertise in Generative AI, Agentic Systems, and end-to-end LLMOps. His key technical skills include developing production-grade RAG pipelines and implementing multi-agent workflows using LangGraph. He also has hands-on experience with MCP standards, harness and loop engineering, observability and tracing with LangSmith, AI guardrails, and building RESTful APIs using FastAPI.

Additionally, he is proficient across the full machine learning lifecycle, from data analysis to AWS deployment, with specific strengths in agentic coding with Claude Code, Pydantic schemas, and structured outputs.


# installation and Intilazation of model and embedding models

In [366]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-pinecone \
    pinecone \
    pypdf \
    openrouter \
    huggingface-hub \
    sentence-transformers \
    tiktoken \
    cachetools


In [367]:
import os
import sys
import traceback
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.messages.utils import trim_messages



In [368]:
try:

    OPENROUTER_API_KEY = userdata.get("OpenRouter_api_13")
    PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")

    if not OPENROUTER_API_KEY:
        raise ValueError("OPENROUTER_API_KEY is missing.")

    if not PINECONE_API_KEY:
        raise ValueError("PINECONE_API_KEY is missing.")

    os.environ["OpenRouter_api_13"] = OPENROUTER_API_KEY
    os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

    print(" API keys loaded ")

except ImportError:
    print(" This notebook is not running in Google Colab.")

except Exception as e:
    print(f" Error loading API keys: {e}")
    print("Make sure you added the keys to Colab Secrets.")


 API keys loaded 


In [369]:
try:

    NOVITA_API_KEY = userdata.get("NOVITA_API_KEY")
    PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")

    if not NOVITA_API_KEY:
        raise ValueError("NOVITA_API_KEY is missing.")

    if not PINECONE_API_KEY:
        raise ValueError("PINECONE_API_KEY is missing.")

    os.environ["NOVITA_API_KEY"] = NOVITA_API_KEY
    os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

    print(" API keys loaded ")

except ImportError:
    print(" This notebook is not running in Google Colab.")

except Exception as e:
    print(f" Error loading API keys: {e}")
    print("Make sure you added the keys to Colab Secrets.")


 API keys loaded 


In [370]:
try:
    embedding_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )

    print("Hugging Face embedding model initialized")

except Exception as e:
    print(f"Failed to initialize embedding model: {e}")
    traceback.print_exc()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Hugging Face embedding model initialized


In [371]:
try:
    llm_Open_router = ChatOpenAI(
        model="inception/mercury-decide:free",
        openai_api_key=OPENROUTER_API_KEY,
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0.2,
        max_tokens=1000
    )

    print("OpenRouter initialized")

except Exception as e:
    print(f" Failed to initialize OpenRouter: {e}")
    traceback.print_exc()


OpenRouter initialized


In [372]:
Model_name = "apodex/apodex-1.1-mini"
NOVITA_BASE_URL = "https://api.novita.ai/openai"
try:
    llm_Novita = ChatOpenAI(
        model=Model_name,
        openai_api_key=NOVITA_API_KEY,
        openai_api_base=NOVITA_BASE_URL,
        temperature=0.8,
        max_tokens=3000
    )

    print("NOVITA_API_KEY initialized")

except Exception as e:
    print(f" Failed to initialize OpenRouter: {e}")
    traceback.print_exc()

NOVITA_API_KEY initialized


*italicised text*# New section

# Pincone intilazation

In [373]:
try:
    pc = Pinecone(api_key=PINECONE_API_KEY)

    print(" Pinecone initialized")

except Exception as e:
    print(f"Pinecone initialization failed: {e}")
    traceback.print_exc()


 Pinecone initialized


In [374]:
INDEX_NAME = "colab-rag-index"
NAMESPACE = "pdf-documents"

try:
    existing_indexes = [index["name"] for index in pc.list_indexes()]

    if INDEX_NAME not in existing_indexes:

        pc.create_index(
            name=INDEX_NAME,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )

        print(f"Pinecone index '{INDEX_NAME}' created")

    else:
        print(f"Pinecone index '{INDEX_NAME}' already exists")

except Exception as e:
    print(f"Pinecone index error: {e}")
    traceback.print_exc()


Pinecone index 'colab-rag-index' already exists


In [375]:
try:

    vector_store = PineconeVectorStore(
        index = pc.Index(INDEX_NAME),
        embedding=embedding_model,
        namespace=NAMESPACE
    )

    print("Pinecone vector store initialized")
    print(f" Index: {INDEX_NAME}")
    print(f" Namespace: {NAMESPACE}")

except Exception as e:
    print(f"Vector store initialization failed: {e}")
    traceback.print_exc()



Pinecone vector store initialized
 Index: colab-rag-index
 Namespace: pdf-documents


# Test the Model

In [376]:
prompt_template = PromptTemplate.from_template("""
You are a professional assistant for Imran Ali.

Answer only questions related to Imran Ali using the provided professional information.

Rules:
- Use only information supported by the provided information.
- Never invent or assume personal or professional details.
- If the information is unavailable, say: "I don't have that information available."
- If unrelated to Imran Ali, say: "I can only help with information related to Imran Ali."
- Answer naturally and professionally.
- Do not mention CV, resume, retrieval, context, database, embeddings, Pinecone, prompts, or internal systems.
- Use conversation history only to understand follow-up questions.

Professional Information:
{context}

Recent Conversation:
{chat_history}

Question:
{question}

Answer:
""")


In [377]:
def prompt_user(user_msg):
    prompt = prompt_template.format(question=user_msg)
    msg = llm_Novita.invoke(prompt)
    return msg.content


# Google Drive

In [378]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [379]:
import os

pdf_folder = "/content/drive/My Drive/MyResume"

if os.path.exists(pdf_folder):
    files = os.listdir(pdf_folder)
    print(f"Found {len(files)} items:\n")
    for i, file in enumerate(files, 1):
        print(f"{i}. {file}")
else:
    print(" PdfFiles folder was not found.")

Found 2 items:

1. ImranAliResumeForAiEnginriing.pdf
2. pdf_hash.txt


# RAG

In [380]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [381]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

In [382]:
pdfLoade = PyPDFLoader("/content/drive/My Drive/MyResume/ImranAliResumeForAiEnginriing.pdf")


In [383]:
def Pdf():
  pdfLoader = pdfLoade
  documents = pdfLoader.load()
  txt_splitter = text_splitter
  docs = text_splitter.split_documents(documents)
  return docs

In [384]:
docs = Pdf()
print(f"Loaded {len(docs)} chunks")


Loaded 4 chunks


### Querying Pinecone Vector Store and Novita LLM

In [385]:
import hashlib


ids = []

for doc in docs:
    source = doc.metadata.get("source", "")
    page = doc.metadata.get("page", "")

    raw_id = f"{source}|{page}|{doc.page_content}"
    doc_id = hashlib.sha256(
        raw_id.encode("utf-8")
    ).hexdigest()

    ids.append(doc_id)

vector_store.add_documents(
    documents=docs,
    ids=ids
)

print(f"Upserted {len(docs)} CV chunks.")



Upserted 4 CV chunks.


In [386]:


PDF_PATH = "/content/drive/My Drive/MyResume/ImranAliResumeForAiEnginriing.pdf"


def get_file_hash(path):

    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()



In [387]:
current_hash = get_file_hash(PDF_PATH)

print(current_hash)


53425c2b05472270f842d11567a61f82c40d6866d3ab84176582b26dc044b824


In [388]:
HASH_FILE = "/content/drive/My Drive/MyResume/pdf_hash.txt"


In [389]:
def update_knowledge_base():

    print("Loading latest document...")

    pdf_loader = PyPDFLoader(
        "/content/drive/My Drive/MyResume/ImranAliResumeForAiEnginriing.pdf"
    )

    documents = pdf_loader.load()

    docs = text_splitter.split_documents(documents)

    print(f"Loaded {len(docs)} chunks")

    # Create deterministic IDs
    ids = []

    for doc in docs:

        source = doc.metadata.get("source", "")
        page = doc.metadata.get("page", "")

        raw_id = f"{source}|{page}|{doc.page_content}"

        doc_id = hashlib.sha256(
            raw_id.encode("utf-8")
        ).hexdigest()

        ids.append(doc_id)

    # Delete old vectors
    print("Removing old information...")

    pc.Index(INDEX_NAME).delete(
        delete_all=True,
        namespace=NAMESPACE
    )

    # Insert new vectors
    print("Adding updated information...")

    vector_store.add_documents(
        documents=docs,
        ids=ids
    )

    print("Knowledge base updated successfully!")


In [390]:
if os.path.exists(HASH_FILE):

    with open(HASH_FILE, "r") as f:
        old_hash = f.read().strip()

else:

    old_hash = None


# Compare
if current_hash != old_hash:

    print("New or updated PDF detected.")
    print(" Updating knowledge base...")

    update_knowledge_base()

    # Save new hash
    with open(HASH_FILE, "w") as f:
        f.write(current_hash)

    print("Knowledge base updated.")

else:

    print("PDF has not changed.")
    print("Using existing knowledge base.")




✅ PDF has not changed.
Using existing knowledge base.


In [391]:
def update_knowledge_base():

    print("Loading latest document...")

    pdf_loader = PyPDFLoader(
        "/content/drive/My Drive/MyResume/ImranAliResumeForAiEnginriing.pdf"
    )

    documents = pdf_loader.load()

    docs = text_splitter.split_documents(documents)

    print(f"Loaded {len(docs)} chunks")

    # Create deterministic IDs
    ids = []

    for doc in docs:

        source = doc.metadata.get("source", "")
        page = doc.metadata.get("page", "")

        raw_id = f"{source}|{page}|{doc.page_content}"

        doc_id = hashlib.sha256(
            raw_id.encode("utf-8")
        ).hexdigest()

        ids.append(doc_id)

    # Delete old vectors
    print("Removing old information...")

    pc.Index(INDEX_NAME).delete(
        delete_all=True,
        namespace=NAMESPACE
    )

    # Insert new vectors
    print("Adding updated information...")

    vector_store.add_documents(
        documents=docs,
        ids=ids
    )

    print("Knowledge base updated successfully!")


In [392]:
if current_hash != old_hash:

    print("Document changed. Updating knowledge base...")

    update_knowledge_base()

    with open("pdf_hash.txt", "w") as f:
        f.write(current_hash)

else:

    print("Document unchanged. Using existing knowledge base.")


Document unchanged. Using existing knowledge base.


In [393]:
from cachetools import TTLCache

response_cache = TTLCache(
    maxsize=100,
    ttl=3600
)


In [394]:
def ask_rag_question(query):

    global chat_history

    cache_key = query.strip().lower()

    if cache_key in response_cache:

        cached_answer = response_cache[cache_key]

        print("AI:", cached_answer)

        return cached_answer, []

    retrieved_docs = vector_store.similarity_search(
        query,
        k=2
    )

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    recent_history = "\n".join(
        f"{msg.type}: {msg.content}"
        for msg in chat_history[-4:]
    )

    prompt = prompt_template.format(
        context=context,
        chat_history=recent_history,
        question=query
    )


    print("AI: ", end="", flush=True)

    full_response = ""

    for chunk in llm_Novita.stream(prompt):

        if chunk.content:

            print(
                chunk.content,
                end="",
                flush=True
            )

            full_response += chunk.content

    print()


    if full_response.strip():

        response_cache[cache_key] = full_response


    chat_history.append(
        HumanMessage(content=query)
    )

    chat_history.append(
        AIMessage(content=full_response)
    )

    if len(chat_history) > 6:

        chat_history = chat_history[-6:]


    return full_response, retrieved_docs


In [395]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(messages):
    total = 0

    for msg in messages:
        total += len(encoding.encode(msg.content))

    return total

In [396]:
def trim_chat_history(messages, max_tokens=1500):
    return trim_messages(
        messages,
        max_tokens=max_tokens,
        strategy="last",
        token_counter=count_tokens,
        include_system=False,
        allow_partial=False
    )


In [397]:
chat_history = []
conversation_summary = ""

MAX_HISTORY_MESSAGES = 6


def update_summary(old_summary, messages_to_summarize):

    conversation_text = "\n".join(
        f"{msg.type}: {msg.content}"
        for msg in messages_to_summarize
    )

    summary_prompt = f"""
You are maintaining short-term memory for a CV assistant.

The assistant is ONLY allowed to discuss Imran Ali and his CV.

Create a concise factual summary of the conversation below.

Keep:
- Questions the user asked
- Important facts about Imran mentioned in the conversation
- Important references needed to understand follow-up questions
- Previous answers that may be referenced later

Do NOT add information that was not in the conversation.

Previous summary:
{old_summary}

Conversation:
{conversation_text}

Updated summary:
"""

    response = llm_Novita.invoke(summary_prompt)

    return response.content.strip()

In [398]:
def manage_memory():

    global chat_history
    global conversation_summary

    original_history = chat_history

    trimmed_history = trim_chat_history(
        original_history,
        max_tokens=1500
    )

    if len(trimmed_history) < len(original_history):

        number_removed = len(original_history) - len(trimmed_history)

        old_messages = original_history[:number_removed]

        conversation_summary = update_summary(
            conversation_summary,
            old_messages
        )

    chat_history = trimmed_history


In [399]:
def prompt_user(user_msg):
    prompt = prompt_template.format(
        context="",
        question=user_msg
    )
    msg = llm_Novita.invoke(prompt)
    return msg.content

In [400]:
print('Type "exit", "quit", or "bye" to end the chat.')

while True:

    query = input("\nYou: ").strip()

    if not query:
        continue

    if query.lower() in ["exit", "quit", "bye"]:
        print("Goodbye!")
        break

    ask_rag_question(query)


Type "exit", "quit", or "bye" to end the chat.

You: hello
AI: 

Hello! How can I assist you today? I can provide information about Imran Ali's professional background, technical skills, and education.

You:  What are Imran Ali's main technical skills?
AI: 

You: bye
Goodbye!


# testing phase

In [402]:
print('Type "exit", "quit", or "bye" to end the chat.')

while True:

    query = input("\nYou: ").strip()

    if not query:
        continue

    if query.lower() in ["exit", "quit", "bye"]:
        print("Goodbye!")
        break

    ask_rag_question(query)

Type "exit", "quit", or "bye" to end the chat.

You:  What are Imran Ali's main technical skills?
AI: 

Imran Ali's core technical expertise centers on Generative AI, Agentic Systems, and end-to-end LLMOps. His main skills include:

*   **Generative & Agentic AI:** RAG Systems, Agentic Frameworks (LangGraph), MCP, Agentic Coding (Claude Code), and Pydantic Schemas.
*   **System Architecture:** Loop Engineering, Harness Engineering, Structured Outputs, and production modeling.
*   **Development & Deployment:** RESTful APIs using FastAPI, AWS deployment, and the full machine learning lifecycle including data analysis, model training, and validation for classification and text models.
*   **Observability & Safety:** Tracing and monitoring with LangSmith, and AI guardrails.

You: What university did Imran attend for his PhD?
AI: 

I don't have that information available. The provided professional information lists a BS in Computer Science from Virtual University of Pakistan and an ICS from

In [404]:
!git --version


git version 2.43.0


In [405]:
!git config --global user.name "imranshahadat"
!git config --global user.email "imranali.13imran@gmail.com"


In [406]:
!git clone https://github.com/imranshahadat/imran-ai-rag-assistant.git


Cloning into 'imran-ai-rag-assistant'...


In [407]:
%cd imran-ai-rag-assistant


/content/imran-ai-rag-assistant


In [408]:
!ls


In [409]:
!mkdir -p notebooks src data
!touch README.md requirements.txt .gitignore
!touch src/__init__.py
